In [1]:
import sys
import shutil
import cv2 as cv
import numpy as np
import pandas as pd
from pathlib import Path

import mediapipe as mp

np.set_printoptions(
    threshold=sys.maxsize  # prevents long arrays from saving to csv with "..."
)

In [2]:
ROOT = Path.cwd().parent

In [3]:
def get_pose_coords(input_vid):
    """Extracts body pose coordinates from a video and write them to a file.

    Args:
        input_vid: Path to the input video file.

    Returns:
        Path to the results file containing pose coordinates.

    Raises:
        ValueError: If the video file cannot be opened.
    """

    input_vid = Path(input_vid)

    pose_directory_path = ROOT / "asl_citation_forms" / "pose_results"
    pose_directory_path.mkdir(parents=True, exist_ok=True)  # parents=True creates parent directories if needed
    result_file_path = pose_directory_path / (input_vid.stem + "_pose-results.txt")  # cuts off extension and adds _pose-results.txt

    mp_pose = mp.solutions.pose

    pose = mp_pose.Pose(
        static_image_mode=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    # open video file (added a raise error here to handle gracefully rather than printing error alongside full trace)
    cap = cv.VideoCapture(str(input_vid))
    if not cap.isOpened():
        raise ValueError(f"Could not open video: {input_vid}")

    try:
        with open(result_file_path, "w") as results_file:  # open the file in write mode and write some content
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break

                image_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)  # convert the BGR image to RGB
                results = pose.process(image_rgb)  # process the image and detect pose landmarks
                
                if results.pose_landmarks:  # if pose landmarks exist, then get image dimensions to convert normalized coordinates to pixel coordinates
                    h, w, c = frame.shape

                    for id, lm in enumerate(results.pose_landmarks.landmark):  # write pose landmarks to file
                        cx, cy = int(lm.x * w), int(lm.y * h)
                        results_file.write(f"{id}, {cx}, {cy}, {lm.z}\n")  # Landmark {id}: x={cx}, y={cy}, z={lm.z}, visibility={lm.visibility}
    
    finally:  # releases resources & closes file
        cap.release()
        pose.close()

    return result_file_path

In [4]:
pose_model_path = ROOT / "pose_landmarker_lite.task"  # set the model

In [5]:
# specify videos to extract pose coords from training data
folder_path = ROOT / "asl_citation_forms"

for video_path in folder_path.glob("*.mp4"):  # can change out for .webm if needed for semlex
    get_pose_coords(video_path)

I0000 00:00:1780586420.633700 1211091 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1780586420.694724 1211155 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780586420.705084 1211153 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780586420.718628 1211163 landmark_projection_calculator.cc:186] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
I0000 00:00:1780586421.936374 1211091 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1780586421.995114 1211215 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference.

In [6]:
# sort training results into subdirectories
pose_path = ROOT / "asl_citation_forms" / "pose_results"

for file_path in folder_path.iterdir():
    if "pose" in file_path.name:
        dest = pose_path / file_path.name
        if dest.exists():  # if already exists, remove & replace
            dest.unlink()
        shutil.move(file_path, pose_path)

In [7]:
# pose coordinates for test chocolate video
test_vid = ROOT / "chocolate-me.mov"
get_pose_coords(test_vid)

I0000 00:00:1780586444.110608 1211091 gl_context.cc:369] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M3 Pro
W0000 00:00:1780586444.172019 1211766 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1780586444.181821 1211766 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


PosixPath('/Users/limey/Desktop/asl_summer26/data_processing/asl_citation_forms/pose_results/chocolate-me_pose-results.txt')

In [8]:
# SEMLEX data import
semlex_data = (pd.read_csv(ROOT / "semlex_metadata.csv")
              .loc[:, ['video_id', 'split', 'label', 'Handshape', 'SignType',
                       'PathMovement', 'RepeatedMovement', 'MajorLocation', 
                       'Contact', 'NondominantHandshape']]
              .query("split == 'train'"))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/limey/Desktop/asl_summer26/data_processing/semlex_metadata.csv'

In [ ]:
# pose results import

def load_pose_data(input_file, pose_dict):
    """Loads pose landmark data from a results file into a dictionary.

    Args:
        input_file: Path to the pose results .txt file.
        pose_dict: Dictionary to load pose data into, keyed by sign name.

    Returns: 
        Updated pose_dict with the new entry added.

    Raises:  
        FileNotFoundError: If the pose results file does not exist.
    """
    input_file = Path(input_file)
    if not input_file.exists():
        raise FileNotFoundError(f"Pose results file not found: {input_file}")
    data = []
    one_frame = []

    with open(input_file, "r") as file:
        for line in file:
            values = [float(x) for x in line.strip().split(", ")]  # lambda here helps reduce the 5-line loop to strip, split, and float each number in file
            landmark_id = values[0]
            coords = values[1:]  # [x, y, z]; skipping first empty one_frame since the first index is 0. 

            if landmark_id == 0:
                if one_frame:
                    data.append(one_frame)
                one_frame = [coords]
            else:
                one_frame.append(coords)

        if one_frame:  # add the last frame just in case
            data.append(one_frame)
        
        key_str = input_file.stem.replace("_pose-results", "")
        if key_str not in pose_dict:
            pose_dict[key_str] = np.array(data)

        return pose_dict

In [ ]:
# load pose data from all result files
folder_path = ROOT / "asl_citation_forms" / "pose_results"
pose_dict = {}
for file_path in folder_path.glob("*.txt"):  # glob helps ignore other files, such as .DS_Store
    load_pose_data(file_path, pose_dict)

In [ ]:
# add pose landmark arrays to SEMLEX data
semlex_data['pose_landmarks'] = None

for i, row in semlex_data.iterrows():
    lemma = row['label']  # use 'LemmaID' for ASL-LEX if needed
    if lemma in pose_dict:
        semlex_data.at[i, 'pose_landmarks'] = pose_dict[lemma]

In [ ]:
# filter semlex to lemmas with landmarks only
filtered_semlex = semlex_data.loc[semlex_data['label'].isin(pose_dict.keys())]
filtered_semlex.head()

,LemmaID,Handshape,SignType,Movement,RepeatedMovement,MajorLocation,MinorLocation,MinorLocation2,Contact,NonDomHandshape,pose_landmarks
3,hamburger,c,SymmetricalOrAlternating,Straight,0.0,Neutral,Neutral,Neutral,1.0,c,"[[[989.0, 275.0, -0.7993969321250916], [1018.0..."
7,cup,c,AsymmetricalDifferentHandshape,Straight,1.0,Hand,Palm,NaN,1.0,B,"[[[973.0, 241.0, -0.5753069519996643], [1001.0..."
8,english,c,AsymmetricalSameHandshape,Straight,1.0,Hand,PalmBack,NaN,1.0,c,"[[[1007.0, 250.0, -0.7348340153694153], [1040...."
11,dentist,s,OneHanded,Straight,1.0,Head,Mouth,NaN,1.0,NaN,"[[[1037.0, 278.0, -0.7727766633033752], [1063...."
14,chair,h,DominanceViolation,Straight,1.0,Hand,FingerBack,NaN,1.0,DominanceConditionViolation,"[[[1032.0, 242.0, -0.8620210886001587], [1062...."


In [ ]:
# filter pose landmarks in 3 representations:
# 1) first 10 frames
# 2) limited pseudo-keyframes (every 10 frames, up to 7 frames total)
# 3) all pseudo-keyframes (every 10 frames)
first_ten_pose_frames = []
pseudokey_n7_frames = []
pseudokey_all_frames = []

for _, row in filtered_semlex.iterrows():
    all_frames = row['pose_landmarks']
    first_ten_pose_frames.append(all_frames[:10])
    pseudokey_n7_frames.append(all_frames[::10][:7])
    pseudokey_all_frames.append(all_frames[::10])

filtered_semlex['pose_first10'] = first_ten_pose_frames
filtered_semlex['pose_pseudokey_n7'] = pseudokey_n7_frames
filtered_semlex['pose_pseudokey_all'] = pseudokey_all_frames

filtered_semlex.head()

/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_51843/3105798811.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['pose_first10'] = first_ten_pose_frames
/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_51843/3105798811.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['pose_pseudokey_n7'] = pseudokey_n7_frames
/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_51843/3105798811.py:17: SettingWithCopyWarning: 
A value is trying to be set o

,LemmaID,Handshape,SignType,Movement,RepeatedMovement,MajorLocation,MinorLocation,MinorLocation2,Contact,NonDomHandshape,pose_landmarks,pose_first10,pose_pseudokey_n7,pose_pseudokey_all
3,hamburger,c,SymmetricalOrAlternating,Straight,0.0,Neutral,Neutral,Neutral,1.0,c,"[[[989.0, 275.0, -0.7993969321250916], [1018.0...","[[[989.0, 275.0, -0.7993969321250916], [1018.0...","[[[989.0, 275.0, -0.7993969321250916], [1018.0...","[[[989.0, 275.0, -0.7993969321250916], [1018.0..."
7,cup,c,AsymmetricalDifferentHandshape,Straight,1.0,Hand,Palm,NaN,1.0,B,"[[[973.0, 241.0, -0.5753069519996643], [1001.0...","[[[973.0, 241.0, -0.5753069519996643], [1001.0...","[[[973.0, 241.0, -0.5753069519996643], [1001.0...","[[[973.0, 241.0, -0.5753069519996643], [1001.0..."
8,english,c,AsymmetricalSameHandshape,Straight,1.0,Hand,PalmBack,NaN,1.0,c,"[[[1007.0, 250.0, -0.7348340153694153], [1040....","[[[1007.0, 250.0, -0.7348340153694153], [1040....","[[[1007.0, 250.0, -0.7348340153694153], [1040....","[[[1007.0, 250.0, -0.7348340153694153], [1040...."
11,dentist,s,OneHanded,Straight,1.0,Head,Mouth,NaN,1.0,NaN,"[[[1037.0, 278.0, -0.7727766633033752], [1063....","[[[1037.0, 278.0, -0.7727766633033752], [1063....","[[[1037.0, 278.0, -0.7727766633033752], [1063....","[[[1037.0, 278.0, -0.7727766633033752], [1063...."
14,chair,h,DominanceViolation,Straight,1.0,Hand,FingerBack,NaN,1.0,DominanceConditionViolation,"[[[1032.0, 242.0, -0.8620210886001587], [1062....","[[[1032.0, 242.0, -0.8620210886001587], [1062....","[[[1032.0, 242.0, -0.8620210886001587], [1062....","[[[1032.0, 242.0, -0.8620210886001587], [1062...."


In [ ]:
filtered_semlex['flat_10frames'] = filtered_semlex['pose_first10'].apply(lambda x: np.array(x).flatten())
filtered_semlex['flat_pseudo_n7'] = filtered_semlex['pose_pseudokey_n7'].apply(lambda x: np.array(x).flatten())
filtered_semlex['flat_pseudo_all'] = filtered_semlex['pose_pseudokey_all'].apply(lambda x: np.array(x).flatten())

# second flatten pass in case of remaining nested arrays
filtered_semlex['flat_10frames'] = filtered_semlex['flat_10frames'].apply(lambda x: x.flatten())
filtered_semlex['flat_pseudo_n7'] = filtered_semlex['flat_pseudo_n7'].apply(lambda x: x.flatten())
filtered_semlex['flat_pseudo_all'] = filtered_semlex['flat_pseudo_all'].apply(lambda x: x.flatten())

/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_51843/4145842384.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['flat_10frames'] = filtered_asl_lex['pose_first10'].apply(lambda x: np.array(x).flatten())
/var/folders/5h/d2lfn8pn1lq5vsgzkwtt18m40000gn/T/ipykernel_51843/4145842384.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_asl_lex['flat_pseudo_n7'] = filtered_asl_lex['pose_pseudokey_n7'].apply(lambda x: np.array(x).flatten())
/var/folders/5h/d2lfn8pn1lq5vsgzkw

In [ ]:
# save to csv for lsh_vid2vid and lsh_frame2frame
filtered_semlex.to_csv("semlex_frame_training_data.csv")